# Ajuste conjunto de edad + masas para una binaria eclipsante joven

isochrone_binary_fit.py

Ajuste conjunto de edad del sistema + masas (M1, M2) de una binaria
eclipsante joven, usando modelos evolutivos/isocronas estelares (PARSEC
o BHAC15/Baraffe15), a partir de:

    Teff1_obs, R1_obs      (primaria; R1 viene del modelado de la curva de luz)
    R2_obs                 (secundaria; viene del modelado de la curva de luz)
    M1_obs (opcional)      (si existe una medición independiente, p. ej. dinámica)

Diseño:
    - M1 y "edad" son los dos parámetros libres que se recorren en una malla 2D.
    - Para cada edad, M2 se obtiene por inversión de R(edad, M2) = R2_obs
      (no es un parámetro libre independiente: queda determinado por R2_obs
      y la edad, dado que ambas estrellas están en la misma isocrona).
    - chi2_total(M1, edad) = chi2_primaria(M1, edad) + chi2_secundaria(edad)
      (+ chi2_masa si hay M1_obs independiente).
    - La verosimilitud sobre la malla se convierte en una posterior por
      remuestreo ponderado (ver `run_posterior_sampling`), que sustituye a un
      MCMC completo -- ver la justificación en el mensaje de análisis.

Ver el notebook `binary_isochrone_fit_demo.ipynb` para un ejemplo de uso
completo con datos de un sistema real.

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator
from scipy.optimize import brentq

T_SUN = 5772.0  # K

## 1. Carga de modelos de isocronas

In [ ]:
def _load_parsec(path_or_dir) -> pd.DataFrame:
    """
    Lee uno o varios archivos de isocronas PARSEC (formato de salida del
    portal CMD) y regresa un DataFrame estandarizado con columnas:
        logAge [log10(yr)], mass [Msun], Teff [K], radius [Rsun], logL [log10(Lsun)]

    Acepta un único archivo que ya contenga múltiples edades (lo usual si se
    pide un rango de edades en CMD) o un directorio con varios archivos de
    una sola edad cada uno (como el 100Myr.dat del código original).
    """
    path = Path(path_or_dir)
    files = [path] if path.is_file() else sorted(path.glob("*.dat"))
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos de isocronas en {path}")

    frames = []
    for f in files:
        header_line = None
        with open(f) as fh:
            for line in fh:
                if line.startswith("#") and "logAge" in line:
                    header_line = line
                    break
        if header_line is None:
            raise ValueError(
                f"No se encontró un encabezado con 'logAge' en {f}. "
                "Revisa que sea un archivo de salida de CMD/PARSEC con la "
                "columna logAge incluida (pide un rango de edades, no una sola)."
            )
        columns = header_line.replace("#", "").split()
        df = pd.read_csv(f, sep=r"\s+", comment="#", names=columns, engine="python")
        df.columns = df.columns.str.strip()
        frames.append(df)

    iso = pd.concat(frames, ignore_index=True)

    required = {"logAge", "Mini", "logTe", "logL"}
    missing = required - set(iso.columns)
    if missing:
        raise ValueError(
            f"Columnas esperadas ausentes en el archivo PARSEC: {missing}. "
            f"Columnas encontradas: {list(iso.columns)}"
        )

    out = pd.DataFrame({
        "logAge": iso["logAge"].astype(float),
        "mass": iso["Mini"].astype(float),  # masa inicial; ojo con pérdida de masa a edades avanzadas
        "Teff": 10.0 ** iso["logTe"].astype(float),
        "logL": iso["logL"].astype(float),
    })
    # PARSEC no da el radio directamente -> se obtiene de L y Teff (Stefan-Boltzmann).
    # OJO: esto hace que "radius" no sea una columna independiente de Teff/logL en
    # este modelo; no combines radius Y logL Y Teff simultáneamente en un chi2
    # para PARSEC, estarías contando la misma información dos veces.
    out["radius"] = np.sqrt(10.0 ** out["logL"]) * (T_SUN / out["Teff"]) ** 2
    out["model"] = "PARSEC"
    return out


def _load_bhac15(path) -> pd.DataFrame:
    """
    Lee el archivo BHAC15_iso.GAIA (Baraffe et al. 2015) y regresa un
    DataFrame estandarizado con las mismas columnas que `_load_parsec`.
    Aquí "radius" SÍ es una salida directa del modelo (columna propia),
    independiente de Teff y logL -- por eso no hay riesgo de doble conteo.
    """
    cols = [
        "mass", "Teff", "logL", "logg", "radius", "Li",
        "F33", "F33B", "F41", "F45B", "F47", "F51", "FHa", "F57", "F63B",
        "F67", "F75", "F78", "F82", "F82B", "F89", "G_RSV", "G", "G_BP", "G_RP",
    ]
    rows, tables = [], []
    current_age_gyr = None

    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("!  t"):
                if rows and current_age_gyr is not None:
                    df = pd.DataFrame(rows, columns=cols)
                    df["logAge"] = np.log10(current_age_gyr * 1e9)  # Gyr -> yr
                    tables.append(df)
                    rows = []
                current_age_gyr = float(line.split("=")[1])
                continue
            if line.startswith("!") or line == "" or line.startswith("BHAC"):
                continue
            if line[0].isdigit():
                values = line.split()
                if len(values) >= len(cols):
                    rows.append([float(v) for v in values[: len(cols)]])

    if rows and current_age_gyr is not None:
        df = pd.DataFrame(rows, columns=cols)
        df["logAge"] = np.log10(current_age_gyr * 1e9)
        tables.append(df)

    if not tables:
        raise ValueError(f"No se pudieron extraer isocronas de {path}. Revisa el formato del archivo.")

    iso = pd.concat(tables, ignore_index=True)
    out = iso[["logAge", "mass", "Teff", "logL", "radius"]].copy()
    out["model"] = "BHAC15"
    return out


_LOADERS = {"PARSEC": _load_parsec, "BHAC15": _load_bhac15}


def load_isochrone_model(model: str, path) -> pd.DataFrame:
    """
    Punto de entrada único para cargar isocronas. `model` es "PARSEC" o
    "BHAC15"; para agregar una familia nueva (p. ej. MIST) basta con
    escribir un loader que regrese las mismas columnas estandarizadas y
    añadirlo a `_LOADERS`, el resto del pipeline no cambia.
    """
    if model not in _LOADERS:
        raise ValueError(f"Modelo '{model}' no reconocido. Opciones: {list(_LOADERS)}")
    grid = _LOADERS[model](path)
    _report_grid_coverage(grid, model)
    return grid


def _report_grid_coverage(grid: pd.DataFrame, model: str) -> None:
    age_min_myr = 10 ** grid["logAge"].min() / 1e6
    age_max_myr = 10 ** grid["logAge"].max() / 1e6
    print(f"[{model}] cobertura de edad:  {age_min_myr:.3g} - {age_max_myr:.3g} Myr")
    print(f"[{model}] cobertura de masa:  {grid['mass'].min():.3f} - {grid['mass'].max():.3f} Msun")

## 2. Interpolación (edad, masa) -> propiedades estelares

In [ ]:
@dataclass
class IsochroneInterpolator:
    grid: pd.DataFrame
    model: str

    def __post_init__(self):
        pts = np.column_stack([self.grid["logAge"].values, self.grid["mass"].values])
        self._teff = LinearNDInterpolator(pts, self.grid["Teff"].values)
        self._radius = LinearNDInterpolator(pts, self.grid["radius"].values)
        self._logL = LinearNDInterpolator(pts, self.grid["logL"].values)
        self.logAge_min, self.logAge_max = float(self.grid["logAge"].min()), float(self.grid["logAge"].max())
        self.mass_min, self.mass_max = float(self.grid["mass"].min()), float(self.grid["mass"].max())

    def teff(self, logAge, mass):
        return self._eval(self._teff, logAge, mass, "Teff")

    def radius(self, logAge, mass):
        return self._eval(self._radius, logAge, mass, "radius")

    def logL(self, logAge, mass):
        return self._eval(self._logL, logAge, mass, "logL")

    def _eval(self, interp, logAge, mass, label):
        logAge_arr = np.atleast_1d(logAge).astype(float)
        mass_arr = np.atleast_1d(mass).astype(float)
        out = np.asarray(interp(logAge_arr, mass_arr), dtype=float)

        out_of_range = (
            (logAge_arr < self.logAge_min) | (logAge_arr > self.logAge_max)
            | (mass_arr < self.mass_min) | (mass_arr > self.mass_max)
        )
        # NaN también aparece cuando el punto cae fuera del casco convexo de
        # los datos aunque esté dentro del rectángulo (edad, masa) nominal
        # (mallas de isocronas no son perfectamente rectangulares).
        outside_hull = np.isnan(out) & ~out_of_range
        if np.any(out_of_range):
            warnings.warn(
                f"{np.sum(out_of_range)} punto(s) fuera del rango cubierto por la malla "
                f"de isocronas ({self.model}) al interpolar '{label}'. Se devuelve NaN "
                "en vez de extrapolar.",
                RuntimeWarning,
            )
        if np.any(outside_hull):
            warnings.warn(
                f"{np.sum(outside_hull)} punto(s) dentro del rango nominal pero fuera "
                f"del casco convexo de la malla ({self.model}, '{label}'); probablemente "
                "una combinación (edad, masa) sin puntos de la malla cerca. Resultado NaN.",
                RuntimeWarning,
            )
        return out if out.size > 1 else out.item()


def interpolate_stellar_properties(interp: IsochroneInterpolator, logAge, mass) -> dict:
    """Envoltura de conveniencia: regresa Teff, radio y logL en un dict."""
    return {
        "Teff": interp.teff(logAge, mass),
        "radius": interp.radius(logAge, mass),
        "logL": interp.logL(logAge, mass),
    }


def invert_radius_to_mass(logAge, R_target, interp: IsochroneInterpolator,
                           mass_bounds: Optional[tuple] = None, n_scan: int = 400) -> float:
    """
    A edad fija, encuentra la masa M tal que radius(logAge, M) = R_target
    mediante búsqueda de raíz (brentq) sobre una malla fina de masas.

    Supone que R(M) es monótona a edad fija dentro del rango de interés.
    Esto es razonable en la pre-secuencia principal para el rango de masas
    típico de estas binarias, pero conviene verificarlo visualmente para tu
    sistema particular (revisa `plot_radius_mass_secondary`).

    Regresa np.nan si R_target no está acotado por (o cae fuera del casco
    convexo dentro de) el rango de masas evaluado -- NO extrapola.
    """
    mlo, mhi = mass_bounds or (interp.mass_min, interp.mass_max)
    m_scan = np.linspace(mlo, mhi, n_scan)
    r_scan = np.atleast_1d(interp.radius(logAge, m_scan))

    valid = np.isfinite(r_scan)
    if valid.sum() < 2:
        return np.nan
    m_scan, r_scan = m_scan[valid], r_scan[valid]

    diff = r_scan - R_target
    sign_changes = np.where(np.diff(np.sign(diff)) != 0)[0]
    if len(sign_changes) == 0:
        return np.nan  # R_target fuera del rango de radios cubierto a esta edad
    i = sign_changes[0]

    def f(m):
        val = interp.radius(logAge, m)
        return (val if np.isfinite(val) else np.nan) - R_target

    try:
        return brentq(f, m_scan[i], m_scan[i + 1])
    except ValueError:
        return np.nan

## 3. Observaciones y chi2

In [ ]:
@dataclass
class BinaryObservables:
    Teff1: float
    sigma_Teff1: float
    R1: float
    sigma_R1: float
    R2: float
    sigma_R2: float
    M1: Optional[float] = None       # solo si hay una medición INDEPENDIENTE del modelo evolutivo
    sigma_M1: Optional[float] = None


def evaluate_binary_model(logAge: float, M1: float, M2: float, interp: IsochroneInterpolator) -> dict:
    """Predicciones del modelo para ambas estrellas a una edad y masas dadas."""
    return {
        "Teff1_model": interp.teff(logAge, M1),
        "R1_model": interp.radius(logAge, M1),
        "Teff2_model": interp.teff(logAge, M2),
        "R2_model": interp.radius(logAge, M2),
    }


def calculate_chi2(logAge: float, M1: float, M2: float, obs: BinaryObservables,
                    interp: IsochroneInterpolator) -> float:
    """
    chi2 conjunto:
        (Teff1_obs - Teff1_model)^2 / sigma_Teff1^2
      + (R1_obs   - R1_model)^2   / sigma_R1^2
      + (R2_obs   - R2_model)^2   / sigma_R2^2
      [+ (M1_obs  - M1)^2         / sigma_M1^2   si hay M1_obs independiente]

    Nota: NO se incluye un término de luminosidad L1 además de Teff1 y R1.
    L = R^2 (Teff/Tsun)^4 es una combinación determinista de R y Teff bajo
    Stefan-Boltzmann; añadir L1_obs como término extra reutilizaría la misma
    información dos veces (ver la discusión de circularidad en el análisis).
    """
    pred = evaluate_binary_model(logAge, M1, M2, interp)
    chi2 = ((obs.Teff1 - pred["Teff1_model"]) / obs.sigma_Teff1) ** 2
    chi2 += ((obs.R1 - pred["R1_model"]) / obs.sigma_R1) ** 2
    chi2 += ((obs.R2 - pred["R2_model"]) / obs.sigma_R2) ** 2
    if obs.M1 is not None:
        chi2 += ((obs.M1 - M1) / obs.sigma_M1) ** 2
    return chi2

## 4. Búsqueda en malla (M1, edad), con M2 determinada por inversión

In [ ]:
@dataclass
class GridSearchResult:
    logAge_grid: np.ndarray
    M1_grid: np.ndarray
    chi2_total: np.ndarray             # shape (n_age, n_M1)
    M2_of_age: np.ndarray              # shape (n_age,) -- M2 no es un eje libre
    chi2_secondary_of_age: np.ndarray  # shape (n_age,)
    best_idx: tuple
    model: str


def run_grid_search(obs: BinaryObservables, interp: IsochroneInterpolator,
                     logAge_range: Optional[tuple] = None, M1_range: Optional[tuple] = None,
                     n_age: int = 200, n_M1: int = 200,
                     enforce_M2_lt_M1: bool = True) -> GridSearchResult:
    """
    Recorre una malla 2D en (edad, M1). Para cada edad, M2 se obtiene por
    inversión de R(edad, M2) = R2_obs (no se recorre como eje independiente:
    ambas estrellas comparten la misma isocrona por construcción).
    """
    logAge_lo, logAge_hi = logAge_range or (interp.logAge_min, interp.logAge_max)
    M1_lo, M1_hi = M1_range or (interp.mass_min, interp.mass_max)

    logAge_grid = np.linspace(logAge_lo, logAge_hi, n_age)
    M1_grid = np.linspace(M1_lo, M1_hi, n_M1)

    M2_of_age = np.full(n_age, np.nan)
    chi2_sec_of_age = np.full(n_age, np.nan)
    for i, logAge in enumerate(logAge_grid):
        M2 = invert_radius_to_mass(logAge, obs.R2, interp)
        M2_of_age[i] = M2
        if not np.isnan(M2):
            R2_model = interp.radius(logAge, M2)
            chi2_sec_of_age[i] = ((obs.R2 - R2_model) / obs.sigma_R2) ** 2

    chi2_total = np.full((n_age, n_M1), np.nan)
    for i, logAge in enumerate(logAge_grid):
        if np.isnan(chi2_sec_of_age[i]):
            continue
        for j, M1 in enumerate(M1_grid):
            if enforce_M2_lt_M1 and M2_of_age[i] >= M1:
                continue  # M2 < M1 violado: combinación físicamente inválida, se descarta
            chi2_total[i, j] = calculate_chi2(logAge, M1, M2_of_age[i], obs, interp)

    if np.all(np.isnan(chi2_total)):
        raise RuntimeError(
            "Ningún punto de la malla produjo un chi2 válido. Verifica que R2_obs "
            "esté dentro del rango de radios del modelo para alguna edad, y que "
            "M2 < M1 sea alcanzable en el rango de edades/masas explorado."
        )

    best_idx = np.unravel_index(np.nanargmin(chi2_total), chi2_total.shape)
    return GridSearchResult(logAge_grid, M1_grid, chi2_total, M2_of_age, chi2_sec_of_age,
                             best_idx, interp.model)

## 5. Posterior por remuestreo de la malla

(sustituye a un MCMC completo -- ver justificación en el análisis)

In [ ]:
def run_posterior_sampling(result: GridSearchResult, n_samples: int = 20000,
                            random_state: Optional[int] = None) -> dict:
    """
    Convierte el mapa de chi2 en una verosimilitud (exp(-chi2/2)), la
    normaliza como densidad de probabilidad sobre la malla (edad, M1), y
    genera muestras de (edad, M1, M2) por remuestreo ponderado.

    Para un problema de baja dimensionalidad (2-3 parámetros) con una malla
    razonablemente fina y acotada, esto da una posterior confiable sin la
    sobrecarga de ajustar un MCMC (cadenas, quemado, mezcla, diagnósticos de
    convergencia). Si en el futuro agregas más parámetros libres (p. ej.
    metalicidad, enrojecimiento) o quieres jerarquizar el ajuste sobre muchas
    binarias a la vez, ahí sí un MCMC (emcee) o nested sampling justifican su
    costo; para este problema no aportan precisión adicional.
    """
    rng = np.random.default_rng(random_state)
    chi2 = result.chi2_total
    finite = np.isfinite(chi2)

    like = np.zeros_like(chi2)
    like[finite] = np.exp(-0.5 * (chi2[finite] - np.nanmin(chi2)))

    prob = like.flatten()
    total = prob.sum()
    if total == 0:
        raise RuntimeError("La verosimilitud es cero en toda la malla; revisa los datos de entrada.")
    prob = prob / total

    flat_idx = rng.choice(len(prob), size=n_samples, p=prob)
    i_idx, j_idx = np.unravel_index(flat_idx, chi2.shape)

    logAge_s = result.logAge_grid[i_idx]
    M1_s = result.M1_grid[j_idx]
    M2_s = result.M2_of_age[i_idx]

    valid = np.isfinite(M2_s)
    return {
        "logAge": logAge_s[valid],
        "age_Myr": 10 ** logAge_s[valid] / 1e6,
        "M1": M1_s[valid],
        "M2": M2_s[valid],
    }


def summarize_posterior(samples: dict) -> dict:
    """Mediana + percentiles 16/84 para cada parámetro muestreado."""
    summary = {}
    for key, values in samples.items():
        p16, p50, p84 = np.percentile(values, [16, 50, 84])
        summary[key] = {"median": p50, "err_low": p50 - p16, "err_high": p84 - p50}
    return summary

## 6. Tabla final y exportación

In [ ]:
def build_results_table(obs: BinaryObservables, result: GridSearchResult,
                         summary: dict, interp: IsochroneInterpolator) -> pd.DataFrame:
    i_best, j_best = result.best_idx
    logAge_best = result.logAge_grid[i_best]
    M1_best = result.M1_grid[j_best]
    M2_best = result.M2_of_age[i_best]
    pred = evaluate_binary_model(logAge_best, M1_best, M2_best, interp)

    rows = [
        ("M1 [Msun]",
         obs.M1 if obs.M1 is not None else np.nan,
         summary["M1"]["median"],
         (obs.M1 - summary["M1"]["median"]) if obs.M1 is not None else np.nan,
         obs.sigma_M1 if obs.sigma_M1 is not None else np.nan),
        ("R1 [Rsun]", obs.R1, pred["R1_model"], obs.R1 - pred["R1_model"], obs.sigma_R1),
        ("Teff1 [K]", obs.Teff1, pred["Teff1_model"], obs.Teff1 - pred["Teff1_model"], obs.sigma_Teff1),
        ("M2 [Msun]", np.nan, summary["M2"]["median"], np.nan, np.nan),
        ("R2 [Rsun]", obs.R2, pred["R2_model"], obs.R2 - pred["R2_model"], obs.sigma_R2),
        ("Edad [Myr]", np.nan, summary["age_Myr"]["median"], np.nan, np.nan),
    ]
    return pd.DataFrame(rows, columns=["Parametro", "Observado", "Modelo", "Diferencia", "Incertidumbre"])


def save_results(table: pd.DataFrame, samples: dict, out_prefix: str) -> None:
    table.to_csv(f"{out_prefix}_tabla.csv", index=False)
    pd.DataFrame(samples).to_csv(f"{out_prefix}_posterior_samples.csv", index=False)
    print(f"Guardado: {out_prefix}_tabla.csv, {out_prefix}_posterior_samples.csv")

## 7. Diagnósticos gráficos

In [ ]:
def _thesis_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 12,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "figure.dpi": 120,
        "savefig.dpi": 300,
    })

In [ ]:
def plot_teff_age_tracks(interp, obs, M1_best, mass_fracs=(0.7, 0.85, 1.0, 1.15, 1.3), save_path=None):
    import matplotlib.pyplot as plt
    _thesis_style()
    logAge_plot = np.linspace(interp.logAge_min, interp.logAge_max, 200)
    age_myr = 10 ** logAge_plot / 1e6

    fig, ax = plt.subplots(figsize=(6, 5))
    for frac in mass_fracs:
        M = float(np.clip(M1_best * frac, interp.mass_min, interp.mass_max))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            teff_track = interp.teff(logAge_plot, np.full_like(logAge_plot, M))
        ax.plot(age_myr, teff_track, label=f"M = {M:.3f} $M_\\odot$")

    ax.axhline(obs.Teff1, color="k", ls="--", lw=1, label="$T_{\\rm eff,1,obs}$")
    ax.fill_between([age_myr.min(), age_myr.max()],
                     obs.Teff1 - obs.sigma_Teff1, obs.Teff1 + obs.sigma_Teff1,
                     color="gray", alpha=0.2)
    ax.set_xscale("log")
    ax.set_xlabel("Edad (Myr)")
    ax.set_ylabel(r"$T_{\rm eff}$ (K)")
    ax.set_title("Primaria: $T_{\\rm eff}$ vs. edad")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

In [ ]:
def plot_radius_age_primary(interp, obs, M1_best, mass_fracs=(0.7, 0.85, 1.0, 1.15, 1.3), save_path=None):
    import matplotlib.pyplot as plt
    _thesis_style()
    logAge_plot = np.linspace(interp.logAge_min, interp.logAge_max, 200)
    age_myr = 10 ** logAge_plot / 1e6

    fig, ax = plt.subplots(figsize=(6, 5))
    for frac in mass_fracs:
        M = float(np.clip(M1_best * frac, interp.mass_min, interp.mass_max))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            r_track = interp.radius(logAge_plot, np.full_like(logAge_plot, M))
        ax.plot(age_myr, r_track, label=f"M = {M:.3f} $M_\\odot$")

    ax.axhline(obs.R1, color="k", ls="--", lw=1, label="$R_{1,\\rm obs}$")
    ax.fill_between([age_myr.min(), age_myr.max()],
                     obs.R1 - obs.sigma_R1, obs.R1 + obs.sigma_R1,
                     color="gray", alpha=0.2)
    ax.set_xscale("log")
    ax.set_xlabel("Edad (Myr)")
    ax.set_ylabel(r"$R_1$ ($R_\odot$)")
    ax.set_title("Primaria: radio vs. edad")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

In [ ]:
def plot_radius_mass_secondary(interp, obs, logAge_best, M2_best, save_path=None):
    import matplotlib.pyplot as plt
    _thesis_style()
    m_grid = np.linspace(interp.mass_min, interp.mass_max, 300)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        r_grid = interp.radius(np.full_like(m_grid, logAge_best), m_grid)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(m_grid, r_grid, color="C0", label=f"isocrona a {10**logAge_best/1e6:.1f} Myr")
    ax.axhline(obs.R2, color="k", ls="--", lw=1, label="$R_{2,\\rm obs}$")
    ax.fill_between([interp.mass_min, interp.mass_max],
                     obs.R2 - obs.sigma_R2, obs.R2 + obs.sigma_R2,
                     color="gray", alpha=0.2)
    ax.axvline(M2_best, color="crimson", ls=":", label=f"$M_2$ = {M2_best:.3f} $M_\\odot$")
    ax.set_xlabel(r"Masa ($M_\odot$)")
    ax.set_ylabel(r"Radio ($R_\odot$)")
    ax.set_title("Secundaria: radio vs. masa a la edad inferida")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

In [ ]:
def plot_chi2_map_primary(result: GridSearchResult, save_path=None):
    import matplotlib.pyplot as plt
    _thesis_style()
    age_myr = 10 ** result.logAge_grid / 1e6
    fig, ax = plt.subplots(figsize=(6.5, 5))
    im = ax.pcolormesh(age_myr, result.M1_grid, result.chi2_total.T, shading="auto", cmap="viridis_r")
    i_best, j_best = result.best_idx
    ax.scatter(age_myr[i_best], result.M1_grid[j_best], color="red", marker="*", s=120, label="mínimo")
    ax.set_xscale("log")
    ax.set_xlabel("Edad (Myr)")
    ax.set_ylabel(r"$M_1$ ($M_\odot$)")
    ax.set_title(r"$\chi^2$ conjunto en el plano $M_1$ vs. edad")
    fig.colorbar(im, ax=ax, label=r"$\chi^2$")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

In [ ]:
def plot_chi2_map_secondary(result: GridSearchResult, save_path=None):
    import matplotlib.pyplot as plt
    _thesis_style()
    age_myr = 10 ** result.logAge_grid / 1e6
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.plot(age_myr, result.chi2_secondary_of_age, color="C1")
    ax2 = ax.twinx()
    ax2.plot(age_myr, result.M2_of_age, color="C2", ls="--")
    ax2.set_ylabel(r"$M_2$ ($M_\odot$)", color="C2")
    ax.set_xscale("log")
    ax.set_xlabel("Edad (Myr)")
    ax.set_ylabel(r"$\chi^2$ secundaria", color="C1")
    ax.set_title(r"Secundaria: $\chi^2$ y $M_2$ vs. edad")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

In [ ]:
def plot_final_isochrone(interp, obs, logAge_best, M1_best, M2_best, save_path=None):
    import matplotlib.pyplot as plt
    _thesis_style()
    m_grid = np.linspace(interp.mass_min, interp.mass_max, 300)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        teff_grid = interp.teff(np.full_like(m_grid, logAge_best), m_grid)
        logL_grid = interp.logL(np.full_like(m_grid, logAge_best), m_grid)

    pred = evaluate_binary_model(logAge_best, M1_best, M2_best, interp)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(teff_grid, 10 ** logL_grid, color="0.5", lw=1.5,
            label=f"isocrona, {10**logAge_best/1e6:.1f} Myr ({interp.model})")
    ax.errorbar(obs.Teff1, 10 ** interp.logL(logAge_best, M1_best),
                xerr=obs.sigma_Teff1, fmt="o", color="crimson", ms=10, capsize=3, label="Primaria (obs.)")
    ax.errorbar(pred["Teff2_model"], 10 ** interp.logL(logAge_best, M2_best),
                fmt="o", color="dodgerblue", ms=10, capsize=3, label="Secundaria (inferida)")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.invert_xaxis()
    ax.set_xlabel(r"$T_{\rm eff}$ (K)")
    ax.set_ylabel(r"Luminosidad ($L_\odot$)")
    ax.set_title("Diagrama HR: isocrona seleccionada")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

In [ ]:
def plot_model_comparison(results_by_model: dict, save_path=None):
    """
    results_by_model: dict {nombre_modelo: dict con llaves 'summary' (de
    summarize_posterior)}. Grafica M1, M2 y edad inferidos por cada familia
    de isocronas para evaluar la sensibilidad del resultado a la elección
    del modelo evolutivo.
    """
    import matplotlib.pyplot as plt
    _thesis_style()
    models = list(results_by_model.keys())
    params = ["M1", "M2", "age_Myr"]
    labels = [r"$M_1$ ($M_\odot$)", r"$M_2$ ($M_\odot$)", "Edad (Myr)"]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, param, label in zip(axes, params, labels):
        meds = [results_by_model[m]["summary"][param]["median"] for m in models]
        lo = [results_by_model[m]["summary"][param]["err_low"] for m in models]
        hi = [results_by_model[m]["summary"][param]["err_high"] for m in models]
        ax.errorbar(range(len(models)), meds, yerr=[lo, hi], fmt="o", capsize=4, color="C0")
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models)
        ax.set_ylabel(label)
    fig.suptitle("Comparación entre familias de isocronas")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    return fig

## 8. Pipeline completo (parametrizable por familia de modelos)

In [ ]:
def run_pipeline(model: str, isochrone_path, obs: BinaryObservables,
                  n_age: int = 200, n_M1: int = 200, n_samples: int = 20000,
                  logAge_range: Optional[tuple] = None, M1_range: Optional[tuple] = None,
                  random_state: int = 0) -> dict:
    grid = load_isochrone_model(model, isochrone_path)
    interp = IsochroneInterpolator(grid, model)
    result = run_grid_search(obs, interp, logAge_range=logAge_range, M1_range=M1_range,
                              n_age=n_age, n_M1=n_M1)
    samples = run_posterior_sampling(result, n_samples=n_samples, random_state=random_state)
    summary = summarize_posterior(samples)
    table = build_results_table(obs, result, summary, interp)
    return {
        "model": model,
        "grid": grid,
        "interp": interp,
        "result": result,
        "samples": samples,
        "summary": summary,
        "table": table,
    }